# Mohamed Eslam — Xception

**DSAI 305 — Phase 2 | Team 505**  
**Task:** Binary Pneumonia Classification on NIH ChestX-ray14

| Item | Detail |
|------|--------|
| Architecture | Xception (Chollet, CVPR 2017) |
| Pretrained weights | ImageNet-1K (timm) |
| CXR reference | Güler & Polat, 2021 |

---
## 1 — Imports

In [1]:
import time
import cv2
import os, time, cv2, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
import timm



import warnings
warnings.filterwarnings('ignore')

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = CUDA_AVAILABLE

print(f'PyTorch: {torch.__version__} | CUDA: {CUDA_AVAILABLE} | Device: {DEVICE}')
if CUDA_AVAILABLE:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from sklearn.metrics import auc as sk_auc
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score,
    precision_score, recall_score, confusion_matrix,
    accuracy_score, classification_report
)

PyTorch: 2.11.0+cu126 | CUDA: True | Device: cuda
GPU: NVIDIA GeForce RTX 3070 Laptop GPU


---
## 2 — Config

In [2]:
RUN_MODE = "full"

IMG_SIZE   = 288
BATCH_SIZE = 16

_EPOCH_MAP = {"smoke": 1, "dev": 15, "full": 50}
NUM_EPOCHS = _EPOCH_MAP[RUN_MODE]

EARLY_STOP_PATIENCE = 20
LABEL_SMOOTHING     = 0.05
MIXUP_ALPHA         = 0.3
EARLY_STOP_ENABLED  = True

# Progressive unfreeze schedule
WARMUP_EPOCHS          = 2
PARTIAL_UNFREEZE_EPOCH = 3
FULL_UNFREEZE_EPOCH    = 6

# Learning rates per stage
HEAD_LR      = 1e-4
PARTIAL_LR   = 2e-5
FULL_LR      = 2e-5
WEIGHT_DECAY = 5e-3

NUM_WORKERS = 0
PIN_MEMORY  = CUDA_AVAILABLE

PROJECT_ROOT = Path('..').resolve()
DATA_SPLITS  = PROJECT_ROOT / 'data' / 'splits'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'Mohamed_Eslam' / 'Xception'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Mode: {RUN_MODE.upper()} | Epochs: {NUM_EPOCHS} | Patience: {EARLY_STOP_PATIENCE}')
print(f'IMG: {IMG_SIZE} | BS: {BATCH_SIZE} | WD: {WEIGHT_DECAY}')
print(f'Unfreeze: A(1-2) B({PARTIAL_UNFREEZE_EPOCH}) C({FULL_UNFREEZE_EPOCH})')

Mode: FULL | Epochs: 50 | Patience: 20
IMG: 288 | BS: 16 | WD: 0.005
Unfreeze: A(1-2) B(3) C(6)


---
## 3 — Data Loading

In [3]:
if RUN_MODE == "full":
    df_train = pd.read_csv(DATA_SPLITS / 'train.csv')
    df_val   = pd.read_csv(DATA_SPLITS / 'val.csv')
    df_test  = pd.read_csv(DATA_SPLITS / 'test.csv')
elif RUN_MODE == "dev":
    df_train = pd.read_csv(DATA_SPLITS / 'train.csv')
    df_val   = pd.read_csv(DATA_SPLITS / 'val.csv')
    df_test  = pd.read_csv(DATA_SPLITS / 'test.csv')
    df_train = df_train.sample(frac=0.2, random_state=42).reset_index(drop=True)
else:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')

# Normalize label column
for _df in [df_train, df_val, df_test]:
    if 'label' in _df.columns and 'target_pneumonia' not in _df.columns:
        _df['target_pneumonia'] = _df['label']
    if 'source_weight' not in _df.columns:
        _df['source_weight'] = 1.0

_pos = 'target_pneumonia'
print(f'Train: {len(df_train):,} | Pos: {df_train[_pos].sum():.0f} | Rate: {df_train[_pos].mean()*100:.1f}%')
print(f'Val  : {len(df_val):,}   | Pos: {df_val[_pos].sum():.0f}   | Rate: {df_val[_pos].mean()*100:.1f}%')
print(f'Test : {len(df_test):,}  | Pos: {df_test[_pos].sum():.0f}  | Rate: {df_test[_pos].mean()*100:.1f}%')


Train: 11,943 | Pos: 1004 | Rate: 8.4%
Val  : 2,611   | Pos: 186   | Rate: 7.1%
Test : 2,618  | Pos: 241  | Rate: 9.2%


---
## 4 — Transforms

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRAIN_RESIZE  = 320

class CLAHETransform:
    """CLAHE with pre-downscale guard — prevents MemoryError on full-res NIH X-rays."""
    def __call__(self, pil_img):
        if max(pil_img.size) > 512:
            pil_img = pil_img.resize((512, 512), Image.LANCZOS)
        img_np   = np.array(pil_img.convert('L'))
        clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(img_np)
        return Image.fromarray(enhanced).convert('RGB')

train_transforms = transforms.Compose([
    CLAHETransform(),
    transforms.Resize(TRAIN_RESIZE),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05),
                            scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.20, contrast=0.20),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    CLAHETransform(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Train: CLAHE->Resize(320)->RandomCrop(288)->Flip->Rotate->Jitter->Norm')
print('Val  : CLAHE->Resize(288)->Norm')

Train: CLAHE->Resize(320)->RandomCrop(288)->Flip->Rotate->Jitter->Norm
Val  : CLAHE->Resize(288)->Norm


---
## 5 — Dataset

In [5]:
class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.image_paths = self.df['image_path'].tolist()
        self.labels = self.df['target_pneumonia'].values.astype(np.float32)
        self.weights = (
            self.df['source_weight'].values.astype(np.float32)
            if 'source_weight' in self.df.columns
            else np.ones(len(self.df), dtype=np.float32)
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label  = self.labels[idx]
        weight = self.weights[idx]
        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (0, 0, 0))
        if self.transform:
            try:
                img = self.transform(img)
            except MemoryError:
                img = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        return img, label, weight

---
## 6 — Model

In [6]:
model = timm.create_model('legacy_xception', pretrained=True)

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.6),
    nn.Linear(num_features, 1)
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: Xception | Params: {total_params:,} | Trainable: {trainable:,} | Device: {DEVICE}')

Model: Xception | Params: 20,809,001 | Trainable: 20,809,001 | Device: cuda


---
## 7 — Loss, Sampler & Optimizer

In [7]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smoothing=0.05):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.smoothing = smoothing

    def forward(self, inputs, targets):
        targets_smooth = targets * (1 - self.smoothing) + \
                         0.5 * self.smoothing
        bce   = F.binary_cross_entropy_with_logits(
                    inputs, targets_smooth, reduction='none'
                )
        pt    = torch.exp(-bce)
        focal = self.alpha * (1.0 - pt) ** self.gamma * bce
        return focal

criterion = FocalLoss(alpha=0.75, gamma=2.0, smoothing=LABEL_SMOOTHING)

# WeightedRandomSampler — 20% positives per batch
TARGET_POS_FRAC = 0.20
# Val/test are now balanced (~7% pos rate after capping)
# 20% positive per batch is a slight oversample — appropriate
labels_array = df_train['target_pneumonia'].values.astype(np.float32)
n_pos_s = int(labels_array.sum())
n_neg_s = int(len(labels_array) - n_pos_s)

sample_weights = np.where(
    labels_array == 1,
    TARGET_POS_FRAC / n_pos_s,
    (1.0 - TARGET_POS_FRAC) / n_neg_s
)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float64),
    num_samples=len(sample_weights), replacement=True
)
train_dataset     = ChestXrayDataset(df_train,     transform=train_transforms)
val_dataset       = ChestXrayDataset(df_val,       transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f'Train loader    : {len(train_dataset):,} samples | {len(train_loader):,} batches')
print(f'Val loader      : {len(val_dataset):,} samples')
print(f'Loss: FocalLoss(0.75, 2.0) | Sampler: {TARGET_POS_FRAC*100:.0f}% pos/batch')
print(f'Train: {len(train_dataset):,} -> {len(train_loader):,} batches')
print(f'Val  : {len(val_dataset):,} -> {len(val_loader):,} batches')
print(f'Stage A: head-only warmup for {WARMUP_EPOCHS} epochs')

Train loader    : 11,943 samples | 747 batches
Val loader      : 2,611 samples
Loss: FocalLoss(0.75, 2.0) | Sampler: 20% pos/batch
Train: 11,943 -> 747 batches
Val  : 2,611 -> 164 batches
Stage A: head-only warmup for 2 epochs


---
## 8 — Train & Validate Functions

In [8]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    n_batches = 0
    for batch_idx, (images, labels, weights) in enumerate(loader):
        images  = images.to(device, non_blocking=True)
        labels  = labels.to(device, non_blocking=True).unsqueeze(1)
        weights = weights.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)
        if MIXUP_ALPHA > 0 and model.training:
            lam      = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
            idx      = torch.randperm(images.size(0), device=images.device)
            images   = lam * images + (1 - lam) * images[idx]
            labels_a, labels_b   = labels, labels[idx]
            weights_a, weights_b = weights, weights[idx]
            outputs    = model(images)
            loss_raw_a = criterion(outputs, labels_a)
            loss_raw_b = criterion(outputs, labels_b)
            loss = (lam    * (loss_raw_a * weights_a) +
                   (1-lam) * (loss_raw_b * weights_b)).mean()
        else:
            outputs  = model(images)
            loss_raw = criterion(outputs, labels)
            loss     = (loss_raw * weights).mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1
    return running_loss / max(n_batches, 1)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n_batches = 0
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels, _ in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            outputs  = model(images)
            loss_raw = criterion(outputs, labels)
            loss     = loss_raw.mean()
            running_loss += loss.item()
            n_batches += 1
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())
    all_labels = np.array(all_labels, dtype=np.float32)
    all_probs  = np.array(all_probs,  dtype=np.float32)
    all_preds  = (all_probs >= 0.5).astype(int)
    return running_loss / max(n_batches, 1), all_preds, all_probs, all_labels

print('Train/validate functions defined.')

Train/validate functions defined.


---
## 9 — Training Loop

In [9]:
PARTIAL_LAYERS = ['block12', 'block11', 'conv4', 'bn4', 'fc']

In [10]:
def tune_threshold(probs, labels):
    thrs = np.arange(0.05, 0.96, 0.05)
    f1s = [f1_score(labels, (probs >= t).astype(int), zero_division=0) for t in thrs]
    best_t = thrs[np.argmax(f1s)]
    thrs = np.arange(max(0.01, best_t - 0.10), min(0.99, best_t + 0.10), 0.01)
    f1s = [f1_score(labels, (probs >= t).astype(int), zero_division=0) for t in thrs]
    best_t = thrs[np.argmax(f1s)]
    thrs = np.arange(max(0.01, best_t - 0.03), min(0.99, best_t + 0.03), 0.005)
    f1s = [f1_score(labels, (probs >= t).astype(int), zero_division=0) for t in thrs]
    return f1s[np.argmax(f1s)], thrs[np.argmax(f1s)]

history = {
    'train_loss': [], 'val_loss': [],
    'val_auc': [], 'val_f1': [],
    'stage': [], 'lr': []
}
best_score        = (-1.0, -1.0)
best_epoch        = 0
epochs_no_improve = 0
best_model_path   = OUTPUT_DIR / 'best_model.pth'
best_checkpoint   = None

print('=' * 70)
print(f'TRAINING XCEPTION | {RUN_MODE.upper()} MODE')
print('=' * 70)

start_time = time.time()

# Stage A: freeze backbone, train head only
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=HEAD_LR, weight_decay=WEIGHT_DECAY
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=WARMUP_EPOCHS, eta_min=1e-6
)
_current_stage = 'A'

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()
    if epoch == PARTIAL_UNFREEZE_EPOCH:
        for param in model.parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(x in name for x in PARTIAL_LAYERS):
                param.requires_grad = True
        trainable_p = [p for p in model.parameters() if p.requires_grad]
        optimizer   = optim.AdamW(trainable_p, lr=PARTIAL_LR, weight_decay=WEIGHT_DECAY)
        remaining   = NUM_EPOCHS - PARTIAL_UNFREEZE_EPOCH + 1
        scheduler   = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining, eta_min=1e-6)
        _current_stage = 'B'

    elif epoch == FULL_UNFREEZE_EPOCH:
        for param in model.parameters():
            param.requires_grad = True
        optimizer  = optim.AdamW(model.parameters(), lr=FULL_LR, weight_decay=WEIGHT_DECAY)
        remaining  = NUM_EPOCHS - FULL_UNFREEZE_EPOCH + 1
        scheduler  = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=6, T_mult=1, eta_min=1e-6
        )
        _current_stage = 'C'

    if _current_stage == 'A': _stage_label = 'STAGE A: Head Warmup'
    elif _current_stage == 'B': _stage_label = 'STAGE B: Partial Fine-tune'
    elif _current_stage == 'C': _stage_label = 'STAGE C: Full Fine-tune'

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_preds_05, val_probs, val_labels = validate(model, val_loader, criterion, DEVICE)

    auc_roc = roc_auc_score(val_labels, val_probs) if val_labels.sum() > 0 else 0.0
    precs_v, recs_v, _ = precision_recall_curve(val_labels, val_probs)
    auc_pr  = sk_auc(recs_v, precs_v)

    acc_05  = accuracy_score(val_labels, val_preds_05)
    prec_05 = precision_score(val_labels, val_preds_05, zero_division=0)
    rec_05  = recall_score(val_labels, val_preds_05, zero_division=0)
    f1_05   = f1_score(val_labels, val_preds_05, zero_division=0)

    best_f1_ep, best_thr_ep = tune_threshold(val_probs, val_labels)
    val_preds_tuned = (val_probs >= best_thr_ep).astype(int)
    prec_tuned = precision_score(val_labels, val_preds_tuned, zero_division=0)
    rec_tuned  = recall_score(val_labels, val_preds_tuned, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(val_labels, val_preds_tuned).ravel()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auc'].append(auc_roc)
    history['val_f1'].append(best_f1_ep)

    dur = time.time() - epoch_start
    lr_curr = optimizer.param_groups[0]['lr']

    print(f'\nEpoch {epoch:2d}/{NUM_EPOCHS} | {_stage_label} | LR: {lr_curr:.2e} | {dur:.0f}s')
    print(f'  LOSS: Train={train_loss:.4f} | Val={val_loss:.4f}')
    print(f'  AUC : ROC={auc_roc:.4f} | PR={auc_pr:.4f}')
    print(f'  @0.5: Acc={acc_05:.4f} | P={prec_05:.4f} | R={rec_05:.4f} | F1={f1_05:.4f} | Pred+={int(val_preds_05.sum())}')
    print(f'  BEST: F1={best_f1_ep:.4f} @ Thr={best_thr_ep:.3f} | P={prec_tuned:.4f} | R={rec_tuned:.4f} | Pred+={int(val_preds_tuned.sum())}')
    print(f'  CONF: TP={tp}, FP={fp}, TN={tn}, FN={fn}')

    current_score = (auc_roc, auc_pr)
    if current_score > best_score:
        best_score = current_score
        best_epoch = epoch
        epochs_no_improve = 0
        best_checkpoint = {'auc_roc': auc_roc, 'auc_pr': auc_pr, 'f1': best_f1_ep,
                           'thr': best_thr_ep, 'prec': prec_tuned, 'rec': rec_tuned,
                           'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn)}
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'val_auc': auc_roc}, best_model_path)
        print(f'  [NEW BEST] Epoch {epoch} | ROC-AUC={auc_roc:.4f} | PR-AUC={auc_pr:.4f}')
    else:
        epochs_no_improve += 1

    if scheduler is not None:
        scheduler.step()
    history['stage'].append(_current_stage)
    history['lr'].append(lr_curr)

    if EARLY_STOP_ENABLED and epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

total_time = (time.time() - start_time) / 60
print(f'\nTraining complete. {total_time:.1f} min. Best epoch: {best_epoch}')

TRAINING XCEPTION | FULL MODE

Epoch  1/50 | STAGE A: Head Warmup | LR: 1.00e-04 | 419s
  LOSS: Train=0.1040 | Val=0.0707
  AUC : ROC=0.5640 | PR=0.0885
  @0.5: Acc=0.9288 | P=0.0000 | R=0.0000 | F1=0.0000 | Pred+=0
  BEST: F1=0.1558 @ Thr=0.400 | P=0.1022 | R=0.3280 | Pred+=597
  CONF: TP=61, FP=536, TN=1889, FN=125
  [NEW BEST] Epoch 1 | ROC-AUC=0.5640 | PR-AUC=0.0885

Epoch  2/50 | STAGE A: Head Warmup | LR: 5.05e-05 | 393s
  LOSS: Train=0.0997 | Val=0.0662
  AUC : ROC=0.5745 | PR=0.0906
  @0.5: Acc=0.9288 | P=0.0000 | R=0.0000 | F1=0.0000 | Pred+=0
  BEST: F1=0.1587 @ Thr=0.380 | P=0.1001 | R=0.3817 | Pred+=709
  CONF: TP=71, FP=638, TN=1787, FN=115
  [NEW BEST] Epoch 2 | ROC-AUC=0.5745 | PR-AUC=0.0906

Epoch  3/50 | STAGE B: Partial Fine-tune | LR: 2.00e-05 | 424s
  LOSS: Train=0.0992 | Val=0.0746
  AUC : ROC=0.6103 | PR=0.1044
  @0.5: Acc=0.9288 | P=0.0000 | R=0.0000 | F1=0.0000 | Pred+=0
  BEST: F1=0.1766 @ Thr=0.420 | P=0.1231 | R=0.3118 | Pred+=471
  CONF: TP=58, FP=413, TN=20

---
## 10 — Training Curves

In [11]:
n_ep = len(history['train_loss'])
if n_ep >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    ep_range = range(1, n_ep + 1)
    axes[0].plot(ep_range, history['train_loss'], 'o-', label='Train', color='#3498db')
    axes[0].plot(ep_range, history['val_loss'], 'o-', label='Val', color='#e74c3c')
    if best_epoch > 0: axes[0].axvline(best_epoch, color='green', ls='--', alpha=0.5, label=f'Best (ep {best_epoch})')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(ep_range, history['val_auc'], 's-', color='#2ecc71')
    if best_epoch > 0: axes[1].axvline(best_epoch, color='green', ls='--', alpha=0.5)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROC-AUC')
    axes[1].set_title('Validation AUC', fontweight='bold'); axes[1].grid(alpha=0.3)
    axes[2].plot(ep_range, history['val_f1'], 'd-', color='#9b59b6')
    if best_epoch > 0: axes[2].axvline(best_epoch, color='green', ls='--', alpha=0.5)
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1-Score')
    axes[2].set_title('Validation F1', fontweight='bold'); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'[SAVED] {OUTPUT_DIR / "training_curves.png"}')

[SAVED] C:\ChestX-ray14\Team505_phase2\outputs\Mohamed_Eslam\Xception\training_curves.png


---
## 11 — Evaluation

In [12]:
checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
val_loss, val_preds_d, val_probs, val_labels = validate(model, val_loader, criterion, DEVICE)

# Threshold tuning
best_thr, best_thr_f1 = 0.5, 0.0
for thr in np.arange(0.05, 0.95, 0.05):
    f1_t = f1_score(val_labels, (val_probs >= thr).astype(int), zero_division=0)
    if f1_t > best_thr_f1:
        best_thr_f1 = f1_t
        best_thr = round(thr, 2)

for thr in np.arange(max(0.01, best_thr-0.10), min(0.99, best_thr+0.10), 0.01):
    f1_t = f1_score(val_labels, (val_probs >= thr).astype(int), zero_division=0)
    if f1_t > best_thr_f1:
        best_thr_f1 = f1_t
        best_thr = round(thr, 2)

val_preds = (val_probs >= best_thr).astype(int)
acc  = accuracy_score(val_labels, val_preds)
prec = precision_score(val_labels, val_preds, zero_division=0)
rec  = recall_score(val_labels, val_preds, zero_division=0)
f1   = f1_score(val_labels, val_preds, zero_division=0)
auc  = roc_auc_score(val_labels, val_probs)

print('=' * 60)
print(f'XCEPTION VALIDATION (epoch {best_epoch}, thr={best_thr})')
print('=' * 60)
print(f'  ROC-AUC   : {auc:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  Accuracy  : {acc:.4f}')
print('=' * 60)

print(classification_report(val_labels, val_preds,
    target_names=['Non-Pneumonia', 'Pneumonia'], digits=4))

XCEPTION VALIDATION (epoch 19, thr=0.35)
  ROC-AUC   : 0.6606
  F1-Score  : 0.1988
  Precision : 0.1365
  Recall    : 0.3656
  Accuracy  : 0.7901
               precision    recall  f1-score   support

Non-Pneumonia     0.9442    0.8227    0.8792      2425
    Pneumonia     0.1365    0.3656    0.1988       186

     accuracy                         0.7901      2611
    macro avg     0.5404    0.5941    0.5390      2611
 weighted avg     0.8866    0.7901    0.8308      2611



In [13]:
cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-Pneumonia', 'Pneumonia'],
    yticklabels=['Non-Pneumonia', 'Pneumonia'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix (Xception, thr={best_thr})\nAcc={acc:.3f}  F1={f1:.3f}  AUC={auc:.3f}',
    fontweight='bold')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'[SAVED] {OUTPUT_DIR / "confusion_matrix.png"}')

[SAVED] C:\ChestX-ray14\Team505_phase2\outputs\Mohamed_Eslam\Xception\confusion_matrix.png


---
## 12 — Save Outputs

In [14]:
metrics_dict = {
    'model': 'Xception', 'mode': RUN_MODE.upper(), 'best_epoch': best_epoch,
    'threshold': best_thr, 'accuracy': round(acc, 4), 'precision': round(prec, 4),
    'recall': round(rec, 4), 'f1_score': round(f1, 4), 'roc_auc': round(auc, 4),
}
pd.DataFrame([metrics_dict]).to_csv(OUTPUT_DIR / 'validation_metrics.csv', index=False)

history_df = pd.DataFrame(history)
history_df.index = history_df.index + 1
history_df.index.name = 'epoch'
history_df.to_csv(OUTPUT_DIR / 'training_history.csv')

print(f'[SAVED] validation_metrics.csv, training_history.csv')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:35s} ({f.stat().st_size/1024:>8.1f} KB)')

[SAVED] validation_metrics.csv, training_history.csv
  .gitkeep                            (     0.0 KB)
  best_model.pth                      ( 81587.6 KB)
  confusion_matrix.png                (    31.6 KB)
  training_curves.png                 (   116.5 KB)
  training_history.csv                (     4.0 KB)
  validation_metrics.csv              (     0.1 KB)


---
## 13 — TTA Evaluation

In [15]:
# TTA Evaluation — 5 augmented views per image
tta_transforms_list = [
    val_transforms,
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(7, 7)), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(degrees=(-7, -7)), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
    transforms.Compose([CLAHETransform(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(brightness=0.15), transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)]),
]

def run_tta_inference(model, df_val, tta_list, device, batch_size=16):
    model.eval()
    all_variant_probs = []
    tta_labels = None
    for v_idx, t in enumerate(tta_list):
        ds = ChestXrayDataset(df_val, transform=t)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
        v_probs, v_labels = [], []
        with torch.no_grad():
            for images, labels, _ in dl:
                images = images.to(device, non_blocking=True)
                probs = torch.sigmoid(model(images)).cpu().numpy().flatten()
                v_probs.extend(probs.tolist())
                if v_idx == 0:
                    v_labels.extend(labels.numpy().flatten().tolist())
        all_variant_probs.append(np.array(v_probs, dtype=np.float32))
        if v_idx == 0:
            tta_labels = np.array(v_labels, dtype=np.float32)
        print(f'  Variant {v_idx} done | mean_prob={np.mean(v_probs):.4f}')
    avg_probs = np.stack(all_variant_probs, axis=0).mean(axis=0)
    return avg_probs, tta_labels

checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'Running TTA with {len(tta_transforms_list)} variants...')

tta_probs, tta_labels = run_tta_inference(model, df_val, tta_transforms_list, DEVICE, BATCH_SIZE)

tta_roc = roc_auc_score(tta_labels, tta_probs)
precs_t, recs_t, _ = precision_recall_curve(tta_labels, tta_probs)
tta_pr = sk_auc(recs_t, precs_t)

best_t_tta, best_f1_tta = 0.5, 0.0
for thr in np.arange(0.05, 0.95, 0.05):
    f1_t = f1_score(tta_labels, (tta_probs >= thr).astype(int), zero_division=0)
    if f1_t > best_f1_tta: best_f1_tta = f1_t; best_t_tta = round(thr, 2)
for thr in np.arange(max(0.01, best_t_tta-0.10), min(0.99, best_t_tta+0.10), 0.01):
    f1_t = f1_score(tta_labels, (tta_probs >= thr).astype(int), zero_division=0)
    if f1_t > best_f1_tta: best_f1_tta = f1_t; best_t_tta = round(thr, 2)

tta_preds = (tta_probs >= best_t_tta).astype(int)
tta_prec = precision_score(tta_labels, tta_preds, zero_division=0)
tta_rec  = recall_score(tta_labels, tta_preds, zero_division=0)
tn, fp, fn, tp = confusion_matrix(tta_labels, tta_preds).ravel()

single_roc = best_checkpoint['auc_roc'] if best_checkpoint else 0.0
single_pr  = best_checkpoint['auc_pr']  if best_checkpoint else 0.0
single_f1  = best_checkpoint['f1']      if best_checkpoint else 0.0

print('\n' + '=' * 70)
print('TTA EVALUATION RESULTS')
print('=' * 70)
print(f'Variants used       : {len(tta_transforms_list)}')
print(f'Val samples         : {len(tta_labels):,}')
print()
print(f'{"Metric":<20} {"Single Checkpoint":>20} {"TTA (5 views)":>15}')
print('-' * 60)
print(f'{"ROC-AUC":<20} {single_roc:>20.4f} {tta_roc:>15.4f}')
print(f'{"PR-AUC":<20} {single_pr:>20.4f} {tta_pr:>15.4f}')
print(f'{"Best F1":<20} {single_f1:>20.4f} {best_f1_tta:>15.4f}')
print('-' * 60)
print(f'TTA Best Threshold  : {best_t_tta:.3f}')
print(f'TTA Precision       : {tta_prec:.4f}')
print(f'TTA Recall          : {tta_rec:.4f}')
print(f'TTA TP={tp} FP={fp} TN={tn} FN={fn}')
print('=' * 70)

tta_metrics = {
    'model': 'Xception', 'evaluation': 'TTA_5variants',
    'best_epoch': best_epoch,
    'tta_roc_auc': round(float(tta_roc), 4), 'tta_pr_auc': round(float(tta_pr), 4),
    'tta_f1': round(float(best_f1_tta), 4), 'tta_threshold': round(float(best_t_tta), 3),
    'tta_precision': round(float(tta_prec), 4), 'tta_recall': round(float(tta_rec), 4),
    'tta_tp': int(tp), 'tta_fp': int(fp), 'tta_tn': int(tn), 'tta_fn': int(fn),
    'single_roc_auc': round(float(single_roc), 4),
    'single_pr_auc': round(float(single_pr), 4),
    'single_f1': round(float(single_f1), 4),
}
with open(OUTPUT_DIR / 'tta_metrics.json', 'w') as f:
    json.dump(tta_metrics, f, indent=2)
print(f'[SAVED] {OUTPUT_DIR / "tta_metrics.json"}')

Running TTA with 5 variants...
  Variant 0 done | mean_prob=0.2501
  Variant 1 done | mean_prob=0.2434
  Variant 2 done | mean_prob=0.2687
  Variant 3 done | mean_prob=0.2469
  Variant 4 done | mean_prob=0.2502

TTA EVALUATION RESULTS
Variants used       : 5
Val samples         : 2,611

Metric                  Single Checkpoint   TTA (5 views)
------------------------------------------------------------
ROC-AUC                            0.6606          0.6633
PR-AUC                             0.1155          0.1160
Best F1                            0.1988          0.2005
------------------------------------------------------------
TTA Best Threshold  : 0.310
TTA Precision       : 0.1280
TTA Recall          : 0.4624
TTA TP=86 FP=586 TN=1839 FN=100
[SAVED] C:\ChestX-ray14\Team505_phase2\outputs\Mohamed_Eslam\Xception\tta_metrics.json
